In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install -q transformers accelerate bitsandbytes torch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 14.8 MB/s eta 0:00:00


In [ ]:
import os
import json
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from tqdm import tqdm

In [ ]:
DATASET_PATH = {
    "reasoning": "/content/drive/MyDrive/Resilio/prompts/reasoning_prompts.json",
    "logical": "/content/drive/MyDrive/Resilio/prompts/logical_prompts.json",
    "classification": "/content/drive/MyDrive/Resilio/prompts/classification_prompts.json",
    "qna": "/content/drive/MyDrive/Resilio/prompts/qna_prompts.json"
}

MODELS = {
    "llama":   "meta-llama/Meta-Llama-3-8B-Instruct",
    "mistral": "mistralai/Mistral-7B-Instruct-v0.2",
    "phi":     "microsoft/Phi-3-mini-4k-instruct",
}

QUANTIZATION_LEVELS = ["fp16", "8bit", "4bit"]

In [ ]:
OUTPUT_ROOT = "/content/drive/MyDrive/Resilio/outputs"
os.makedirs(OUTPUT_ROOT, exist_ok=True)

In [ ]:
def load_model(model_name, quant_level):
    print(f"\nLoading {model_name} with {quant_level}...")

    tokenizer = AutoTokenizer.from_pretrained(model_name)
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "left"

    # Common kwargs for all loading types
    model_kwargs = {
        "device_map": "auto",
        "trust_remote_code": True
    }

    if quant_level == "fp16":
        model = AutoModelForCausalLM.from_pretrained(
            model_name,
            torch_dtype=torch.float16,
            **model_kwargs
        )

    elif quant_level == "8bit":
        # Wrap 8-bit logic in BitsAndBytesConfig
        quant_config = BitsAndBytesConfig(load_in_8bit=True)
        model = AutoModelForCausalLM.from_pretrained(
            model_name,
            quantization_config=quant_config,
            **model_kwargs
        )

    elif quant_level == "4bit":
        # Wrap 4-bit logic in BitsAndBytesConfig
        quant_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_use_double_quant=True
        )
        model = AutoModelForCausalLM.from_pretrained(
            model_name,
            quantization_config=quant_config,
            **model_kwargs
        )

    else:
        raise ValueError("Invalid quantization level. Choose from 'fp16', '8bit', or '4bit'.")

    return tokenizer, model

In [ ]:
temp = 0.5
top_p = 0.9
max_token = 384
rep_pen = 1.2

SYSTEM_PROMPT = """You are a knowledgeable assistant. Answer the question directly and concisely.
You are a precise and concise AI assistant.
Rules:
- The response MUST be under 256 tokens.
- Aim for 150–220 tokens when possible.
- Do not exceed 256 tokens under any circumstances.
- No filler, no repetition, no unnecessary explanations.
- Be direct, structured, and information-dense.
- Use short paragraphs or bullet points when helpful.
- Do not restate the question.
- Do not include disclaimers unless explicitly required.
- End immediately after completing the answer.
If the answer would exceed 256 tokens, summarize aggressively to stay within limit.
"""

def generate_response(tokenizer, model, prompt):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": prompt}
    ]

    formatted = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(formatted, return_tensors="pt")
    input_len = inputs["input_ids"].shape[1]
    inputs = inputs.to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_token,
            temperature=temp,
            top_p=top_p,
            do_sample=True,
            repetition_penalty=rep_pen,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id
        )
    response = outputs[0][input_len:]
    return tokenizer.decode(response, skip_special_tokens=True).strip()


# changes below

In [ ]:
MODEL = MODELS["mistral"]   # llama, mistral, phi
QUANT = QUANTIZATION_LEVELS[2]  #0-fp16, 1-8bit, 2-4bit

tokenizer, model = load_model(MODEL, QUANT)

output_folder = os.path.join(OUTPUT_ROOT, f"{MODEL}_{QUANT}")
os.makedirs(output_folder, exist_ok=True)


Loading mistralai/Mistral-7B-Instruct-v0.2 with 4bit...


config.json:   0%|          | 0.00/596 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

In [ ]:
task_category = "reasoning"     # reasoning, logical, classification, qna


DATASET_PATH = DATASET_PATH[task_category]
with open(DATASET_PATH, "r") as f:
    dataset = json.load(f)

prompts = dataset["prompts"]

print(f"Loaded {len(prompts)} prompts")


Loaded 160 prompts


In [ ]:
output_folder_for_task = os.path.join(output_folder, f"{task_category}")
os.makedirs(output_folder_for_task, exist_ok=True)

checkpoint_path = os.path.join(output_folder_for_task, "responses.jsonl")
final_json_path = os.path.join(output_folder_for_task, "responses.json")

for item in tqdm(prompts):
    prompt_text = item["prompt"]

    try:
        response = generate_response(tokenizer, model, prompt_text)
    except Exception as e:
        response = f"ERROR: {str(e)}"
    print(f"-> {response}")

    # Save immediately to prevent data loss
    with open(checkpoint_path, "a") as f:
        result = {**item, "model": MODEL, "quantization": QUANT, "response": response}
        f.write(json.dumps(result) + "\n")

print(f"Process complete. Responses saved to {checkpoint_path}")


final_results = []
try:
    with open(checkpoint_path, "r") as f:
        for line in f:
            final_results.append(json.loads(line))

    with open(final_json_path, "w") as f:
        json.dump(final_results, f, indent=4)

    print(f"\n✅ All responses processed and saved to a clean JSON list at: {final_json_path}")

    # Optional: Remove the .jsonl checkpoint if you only want the final JSON
    # os.remove(checkpoint_path)

except Exception as e:
    print(f"Conversion failed, but your data is safe in {checkpoint_path}. Error: {e}")


  0%|          | 0/160 [00:00<?, ?it/s]

-> Government's decision between renewable energy (RE) and fossil fuel-based plants (FFB):

Economic Factors:
- RE has decreasing costs: Solar & wind prices dropped by 85% & 49%, respectively, since 2009.
- RE creates jobs: Each GW of solar PV capacity generates about 37,000 jobs vs. 700 jobs per GW for coal.
- RE reduces electricity bills: Consumers save on average $10 monthly with rooftop solar.

Environmental Factors:
- FFB contributes significantly to greenhouse gas emissions and air pollution.
- RE produces minimal carbon footprint during operation but may have high embodied energy from manufacturing.

Long-Term Sustainability Factors:
- Renewables offer abundant resources; sun, wind, water, etc., while fossil fuels will eventually run out.
- RE reduces dependence on imported oil and natural gas, improving energy security.

Decision Criteria:
1. Cost-effectiveness: Compare total cost of ownership (installation + maintenance) for both options over their lifetimes.
2. Environmental 

  1%|▏         | 2/160 [01:19<1:45:31, 40.07s/it]

-> To make an informed decision between investing in renewable energy infrastructure vs. expanding fossil fuel-plants, consider the following factors:

Economic Factors:
1. Cost: Renewables have decreasing costs with technology advancements, while fossil fuels may face rising prices due to depleting resources.
2. Job Creation: Both sectors create jobs; however, renewables generate more per unit of capacity than traditional plants.
3. Energy Independence: Renewables can reduce reliance on imported fossil fuels, saving money and ensuring energy security.
4. Grid Stability: Intermittent nature of renewables requires backup solutions like batteries or hydroelectric storage or flexible grid management systems.

Environmental Factors:
1. Greenhouse Gas Emissions: Fossil fuels emit significant amounts of CO2 during combustion, contributing to climate change. Renewables produce little to zero emissions.
2. Water Usage: Thermoelectric power plants require vast quantities of water for cooling, w

  2%|▏         | 3/160 [01:49<1:33:02, 35.56s/it]

...

-> To decide on adopting an emerging technology in an organization, consider these steps:

1. Assess the potential impact: Evaluate how the technology could benefit your business goals, improve processes, or create new opportunities.
2. Market analysis: Research market trends, competitors' adoption rates, and industry predictions about the technology's future growth.
3. Cost-benefit analysis: Determine the financial implications of implementing the technology, including costs, savings, and return on investment (ROI).
4. Technical feasibility: Ensure the technology aligns with existing infrastructure, compatibility with other systems, and ease of integration.
5. Risk assessment: Identify potential risks associated with the technology, such as security concerns, regulatory compliance, and ethical issues.
6. Training requirements: Consider employee training needs and resources necessary to effectively use the technology.
7. Vendor evaluation: Thoroughly research vendors, their reputation,

 99%|█████████▉| 159/160 [1:14:36<00:21, 21.49s/it]

-> To determine if an organization should adopt an emerging technology, consider these factors:

1. Alignment with business strategy: Evaluate if the tech fits long-term goals and objectives.
2. Competitive advantage: Assess potential competitive edge gained by adopting the tech.
3. Cost vs. benefit analysis: Weigh investment against expected returns.
4. Technical feasibility: Ensure compatibility with existing systems and infrastructure.
5. Risk assessment: Identify risks associated with adoption and mitigation strategies.
6. Vendor support and partnerships: Research availability of reliable vendor support and industry collaborations.
7. Training requirements: Plan for employee training on new tools and processes.
8. Regulatory compliance: Understand any legal or regulatory implications.
9. Scalability: Consider future growth prospects and adaptability of the tech.


100%|██████████| 160/160 [1:14:54<00:00, 28.09s/it]

-> To determine if an organization should adopt an emerging technology, consider the following factors:

1. Alignment with business strategy: Does adopting the tech support your long-term goals?
2. Competitive advantage: Will it give you an edge over competitors?
3. Cost vs. benefit: What's the potential ROI compared to implementation costs?
4. Technical feasibility: Can your current infrastructure support it?
5. Security risks: Are there potential vulnerabilities and how can they be mitigated?
6. Training needs: How will employees need to adapt?
7. Vendor reliability: Is the tech stable and supported by a reputable provider?
8. Regulatory compliance: Does adoption comply with industry standards and regulations?
9. Future scalability: Will the tech grow with your organization?

Assessing these factors through careful analysis and consultation with experts can help inform your decision.
Process complete. Responses saved to /content/drive/MyDrive/Resilio/outputs/mistralai/Mistral-7B-Inst

In [ ]:
# Memory Cleanup
del model
del tokenizer
torch.cuda.empty_cache()